# Densenet121 training with CONNIE image dataset

In [1]:
%run ./../notebook_init.py

import os
import torch
import optuna
import mlflow

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import Subset
from itertools import product
from pathlib import Path
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

from core import DATA_FOLDER, RESULTS_FOLDER

from scripts.connie_training_utils import ModelTraining, TransformedSubset, \
    Seed, IMG_SIZE, NPYFolderDataset, densenet121_model

Load file paths and set the computation device to GPU if available; otherwise, use CPU, and initialize the random seed

In [2]:
#train_data = os.path.join(DATA_FOLDER, "train_data_png_full")
train_data = os.path.join(DATA_FOLDER, "train_data_npy_full")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
seed = Seed()

cuda:0


Compute dataset mean and standard deviation, then define training and test transforms with normalization

In [3]:
#basic_transform = transforms.Compose([
#    transforms.Resize(IMG_SIZE),
#    transforms.ToTensor()
#])

# Load dataset without transform
#full_dataset_transform = datasets.ImageFolder(train_data,
#                                              transform=basic_transform)

#mean, std = calculate_mean_std(full_dataset_transform)
#test_transform = get_test_transform(mean, std)
#train_transform = get_train_transform(mean, std)

Split train + validation and test set

In [4]:
#trainval_dataset = datasets.ImageFolder(train_data)
#trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))
trainval_dataset = NPYFolderDataset(train_data)
trainval_set = Subset(trainval_dataset, list(range(len(trainval_dataset))))

## Training with K-fold

In [5]:
# Define your parameter grid
param_grid = {
    "learning_rate": [5e-4],
    'weight_decay': [5e-5],
    "step_size": [10],
    "gamma": [0.5]
}

# Create all combinations
grid = list(product(
    param_grid["learning_rate"],
    param_grid["weight_decay"],
    param_grid["step_size"],
    param_grid["gamma"]
))

k_folds = 5
num_epochs = 100

class_idx_map = trainval_set.dataset.class_to_idx
print("Classes index:", class_idx_map)

Classes index: {'Blob': 0, 'Diffusion Hit': 1, 'Electron': 2, 'Muon': 3, 'Others': 4}


## Hyperparameters Tuning

In [6]:
from pathlib import Path
mlflow.set_tracking_uri(Path(DATA_FOLDER) / "mlruns")
#mlflow.set_tracking_uri(os.path.join(DATA_FOLDER, "mlruns"))

def objective_densenet121(trial, current_class_idx, current_class_name):
    k_folds = 5
    num_epochs = 100
    model_training = ModelTraining()

    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    wd = trial.suggest_float("wd", 1e-5, 1e-2, log=True)
    step_size = trial.suggest_int("step", 5, 50)
    gamma = trial.suggest_float("gamma", 0.1, 0.9)

    hyperparam = {"lr": lr, "wd": wd, "step": step_size, "gamma": gamma}

    with mlflow.start_run(nested=True, run_name=f"DenseNet_trial_{trial.number}"):
        mlflow.set_tag("model_type", "DenseNet-121")
        mlflow.set_tag("kfold_splits", k_folds)
        mlflow.log_params(hyperparam)

        metrics = model_training.train_model_kfold(
            device=device,
            dataset=trainval_set,
            num_epochs=num_epochs,
            k_folds=k_folds,
            current_class_idx=current_class_idx,
            seed=seed,
            model=densenet121_model,
            hyperparam=hyperparam
        )

        mlflow.log_metrics({
            "mean_train_accuracy": metrics["mean_train_accuracy"],
            "std_train_accuracy": metrics["std_train_accuracy"],
            "mean_train_loss": metrics["mean_train_loss"],
        
            "mean_val_accuracy": metrics["mean_val_accuracy"],
            "std_val_accuracy": metrics["std_val_accuracy"],
            "mean_val_loss": metrics["mean_val_loss"],
            "std_val_loss": metrics["std_val_loss"],
        
            "mean_precision": metrics["mean_precision"],
            "std_precision": metrics["std_precision"],
            "mean_recall": metrics["mean_recall"],
            "std_recall": metrics["std_recall"],
            "mean_f1_binary": metrics["mean_f1_binary"],
            "std_f1_binary": metrics["std_f1_binary"],
            "mean_f1_macro": metrics["mean_f1_macro"],
            "std_f1_macro": metrics["std_f1_macro"]
        })

        best_epochs = metrics.get("best_epochs_per_fold")
        if best_epochs is not None:
            mlflow.log_metric("best_epoch_mean", float(np.mean(best_epochs)))
            mlflow.log_metric("best_epoch_std", float(np.std(best_epochs)))
            mlflow.log_metric("best_epoch_median", float(np.median(best_epochs)))
            
        mlflow.log_dict(metrics, "full_metrics.json")

        if "all_val_true" in metrics and "all_val_preds" in metrics:
            all_val_true = metrics["all_val_true"]
            all_val_preds = metrics["all_val_preds"]

            metrics_dir = os.path.join(RESULTS_FOLDER,
                                       "metrics_densenet121", 
                                       current_class_name)
            os.makedirs(metrics_dir, exist_ok=True)
            
            binary_target_names = [f"not_{current_class_name}", current_class_name]

            # === Classification report ===
            report = classification_report(
                all_val_true,
                all_val_preds,
                target_names=binary_target_names,
                output_dict=True,
                zero_division=0
            )
            report_df = pd.DataFrame(report).transpose()

            report_path = os.path.join(metrics_dir,
                                       f"trial_{trial.number}_classification_report.csv")
            report_df.to_csv(report_path)
            mlflow.log_artifact(report_path)

            # === Confusion matrix ===
            cm = confusion_matrix(
                all_val_true,
                all_val_preds,
                labels=[0, 1])

            fig, ax = plt.subplots(figsize=(8, 6))
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                xticklabels=binary_target_names,
                yticklabels=binary_target_names,
                ax=ax
            )
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            fig.tight_layout()

            cm_path = os.path.join(metrics_dir,
                                   f"trial_{trial.number}_confusion_matrix.png")
            fig.savefig(cm_path)
            mlflow.log_artifact(cm_path)
            plt.close(fig)

        return metrics["mean_f1_binary"]


In [ ]:
class_idx_map = trainval_set.dataset.class_to_idx

for class_name, current_class_idx in class_idx_map.items():
    print(f"\n=== Optuna tuning for class '{class_name}' (ID={current_class_idx}) ===")
    # Create OvA binary labels for current class
    #trainval_set.dataset.samples = [
     #   (path, 1 if label == current_class_idx else 0)
     #   for path, label in trainval_set.dataset.samples]
    mlflow.set_experiment(f"Tuning_Densenet121_{class_name}_npy_fixed_2")
    study_densenet121 = optuna.create_study(direction="maximize")
    study_densenet121.optimize(lambda trial: objective_densenet121(trial,
                                                                   current_class_idx,
                                                                   class_name), n_trials=100)

    print(f"Best trials for Densenet121, class '{class_name}':")
    for i, t in enumerate(study_densenet121.best_trials):
        print(f"Trial #{t.number}")
        print(f"  Values (Val Accuracy, Val Loss): {t.values}")
        print(f"  Params: ")
        for key, value in t.params.items():
            print(f"    {key}: {value}")


=== Optuna tuning for class 'Blob' (ID=0) ===


2026/04/08 14:43:47 INFO mlflow.tracking.fluent: Experiment with name 'Tuning_Densenet121_Blob_npy_fixed_2' does not exist. Creating a new experiment.
[I 2026-04-08 14:43:47,625] A new study created in memory with name: no-name-cf0dd0c7-44ff-4e4f-b0c9-80f1fb07ae23



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.4634 Acc: 0.8738
Val Loss: 0.4713 Acc: 0.8547
Val Precision: 0.4024 Recall: 0.9577 F1_binary: 0.5667 F1_macro: 0.7397

Epoch 2/100 — Fold 1
----------
Train Loss: 0.2472 Acc: 0.9521
Val Loss: 0.2012 Acc: 0.9791
Val Precision: 0.8784 Recall: 0.9155 F1_binary: 0.8966 F1_macro: 0.9424

Epoch 3/100 — Fold 1
----------
Train Loss: 0.2293 Acc: 0.9549
Val Loss: 0.1002 Acc: 0.9818
Val Precision: 0.8452 Recall: 1.0000 F1_binary: 0.9161 F1_macro: 0.9530

Epoch 4/100 — Fold 1
----------
Train Loss: 0.1962 Acc: 0.9619
Val Loss: 0.1044 Acc: 0.9707
Val Precision: 0.7717 Recall: 1.0000 F1_binary: 0.8712 F1_macro: 0.9273

Epoch 5/100 — Fold 1
----------
Train Loss: 0.1830 Acc: 0.9581
Val Loss: 0.1536 Acc: 0.9483
Val Precision: 0.6574 Recall: 1.0000 F1_binary: 0.7933 F1_macro: 0.8819

Epoch 6/100 — Fold 1
----------
Train Loss: 0.2140 Acc: 0.9507
Val Loss: 0.1145 Acc: 0.9721
Val Precision: 0.7802 Recall: 

[I 2026-04-08 15:07:31,834] Trial 0 finished with value: 0.8960650672883966 and parameters: {'lr': 0.00011397914433858185, 'wd': 1.2299498590488678e-05, 'step': 17, 'gamma': 0.8150578887336543}. Best is trial 0 with value: 0.8960650672883966.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.5660 Acc: 0.8350
Val Loss: 1.1037 Acc: 0.3939
Val Precision: 0.1406 Recall: 1.0000 F1_binary: 0.2465 F1_macro: 0.3698

Epoch 2/100 — Fold 1
----------
Train Loss: 0.2771 Acc: 0.9525
Val Loss: 0.1802 Acc: 0.9763
Val Precision: 0.8068 Recall: 1.0000 F1_binary: 0.8931 F1_macro: 0.9399

Epoch 3/100 — Fold 1
----------
Train Loss: 0.2245 Acc: 0.9560
Val Loss: 0.1332 Acc: 0.9791
Val Precision: 0.8256 Recall: 1.0000 F1_binary: 0.9045 F1_macro: 0.9463

Epoch 4/100 — Fold 1
----------
Train Loss: 0.2065 Acc: 0.9616
Val Loss: 0.1834 Acc: 0.9385
Val Precision: 0.6174 Recall: 1.0000 F1_binary: 0.7634 F1_macro: 0.8641

Epoch 5/100 — Fold 1
----------
Train Loss: 0.1940 Acc: 0.9609
Val Loss: 0.0991 Acc: 0.9777
Val Precision: 0.8161 Recall: 1.0000 F1_binary: 0.8987 F1_macro: 0.9431

Epoch 6/100 — Fold 1
----------
Train Loss: 0.1964 Acc: 0.9560
Val Loss: 0.1260 Acc: 0.9763
Val Precision: 0.8140 Recall: 

...


==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.3659 Acc: 0.9857
Val Loss: 2.5000 Acc: 0.5754
Val Precision: 0.0288 Recall: 1.0000 F1_binary: 0.0559 F1_macro: 0.3910

Epoch 2/100 — Fold 1
----------
Train Loss: 0.1419 Acc: 0.9703
Val Loss: 0.0177 Acc: 0.9958
Val Precision: 0.7500 Recall: 1.0000 F1_binary: 0.8571 F1_macro: 0.9275

Epoch 3/100 — Fold 1
----------
Train Loss: 0.2538 Acc: 0.9696
Val Loss: 0.1377 Acc: 0.9777
Val Precision: 0.3600 Recall: 1.0000 F1_binary: 0.5294 F1_macro: 0.7590

Epoch 4/100 — Fold 1
----------
Train Loss: 0.1347 Acc: 0.9867
Val Loss: 0.1228 Acc: 0.9693
Val Precision: 0.2903 Recall: 1.0000 F1_binary: 0.4500 F1_macro: 0.7171

Epoch 5/100 — Fold 1
----------
Train Loss: 0.0422 Acc: 0.9944
Val Loss: 0.0210 Acc: 1.0000
Val Precision: 1.0000 Recall: 1.0000 F1_binary: 1.0000 F1_macro: 1.0000

Epoch 6/100 — Fold 1
----------
Train Loss: 0.0269 Acc: 0.9972
Val Loss: 0.0132 Acc: 0.9986
Val Precision: 0.9000 Recall: 

[I 2026-04-11 05:24:14,333] Trial 62 finished with value: 0.9374416433239963 and parameters: {'lr': 0.0006585483871568799, 'wd': 0.00018238072002317476, 'step': 34, 'gamma': 0.30274074616125607}. Best is trial 54 with value: 0.9554867561059511.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.3879 Acc: 0.9654
Val Loss: 1.5841 Acc: 0.6899
Val Precision: 0.0390 Recall: 1.0000 F1_binary: 0.0750 F1_macro: 0.4444

Epoch 2/100 — Fold 1
----------
Train Loss: 0.1390 Acc: 0.9745
Val Loss: 5.9396 Acc: 0.9874
Val Precision: 0.0000 Recall: 0.0000 F1_binary: 0.0000 F1_macro: 0.4968

Epoch 3/100 — Fold 1
----------
Train Loss: 0.4212 Acc: 0.9570
Val Loss: 0.0977 Acc: 0.9818
Val Precision: 0.4091 Recall: 1.0000 F1_binary: 0.5806 F1_macro: 0.7857

Epoch 4/100 — Fold 1
----------
Train Loss: 0.1046 Acc: 0.9867
Val Loss: 0.1223 Acc: 0.9721
Val Precision: 0.3103 Recall: 1.0000 F1_binary: 0.4737 F1_macro: 0.7297

Epoch 5/100 — Fold 1
----------
Train Loss: 0.1048 Acc: 0.9857
Val Loss: 0.0779 Acc: 0.9902
Val Precision: 0.5625 Recall: 1.0000 F1_binary: 0.7200 F1_macro: 0.8575

Epoch 6/100 — Fold 1
----------
Train Loss: 0.0292 Acc: 0.9955
Val Loss: 0.0033 Acc: 1.0000
Val Precision: 1.0000 Recall: 

[I 2026-04-11 05:36:55,281] Trial 63 finished with value: 0.9346638655462185 and parameters: {'lr': 0.0008455903106928105, 'wd': 8.683308575333765e-05, 'step': 37, 'gamma': 0.445248956036743}. Best is trial 54 with value: 0.9554867561059511.



==================== Fold 1/5 ====================

Epoch 1/100 — Fold 1
----------
Train Loss: 0.4304 Acc: 0.9151
Val Loss: 1.1136 Acc: 0.6131
Val Precision: 0.0315 Recall: 1.0000 F1_binary: 0.0610 F1_macro: 0.4087

Epoch 2/100 — Fold 1
----------
Train Loss: 0.4266 Acc: 0.9374
Val Loss: 0.1637 Acc: 0.9735
Val Precision: 0.3214 Recall: 1.0000 F1_binary: 0.4865 F1_macro: 0.7364

Epoch 3/100 — Fold 1
----------
Train Loss: 0.0819 Acc: 0.9878
Val Loss: 0.0677 Acc: 0.9832
Val Precision: 0.4286 Recall: 1.0000 F1_binary: 0.6000 F1_macro: 0.7957

Epoch 4/100 — Fold 1
----------
Train Loss: 0.0366 Acc: 0.9937
Val Loss: 0.0081 Acc: 0.9972
Val Precision: 0.8182 Recall: 1.0000 F1_binary: 0.9000 F1_macro: 0.9493

Epoch 5/100 — Fold 1
----------
Train Loss: 0.0409 Acc: 0.9958
Val Loss: 0.1096 Acc: 0.9832
Val Precision: 0.4286 Recall: 1.0000 F1_binary: 0.6000 F1_macro: 0.7957

Epoch 6/100 — Fold 1
----------
Train Loss: 0.0403 Acc: 0.9927
Val Loss: 0.0093 Acc: 0.9986
Val Precision: 0.9000 Recall: 

In [8]:
best_trial = max(trials, key=lambda x: val_f1)

NameError: name 'trials' is not defined